# Population disaggregation based on building footprint data (filtered through house-or-not classes)
- Filter out building footprints that are not classified as **"house"** by the model
- Disaggregating city-wide data to each building footprint (allocated based on each building footprint's land area)
- Aggregated building footprint population estimate data to barangay-level

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter("ignore", UserWarning)

In [2]:
bldg = gpd.read_file("..//data//ncr//buildings.gpkg", layer="bldg").to_crs(crs="EPSG:32651")
bldg_class = pd.read_csv("..//data//ncr//bldg_class.csv")
bldg = bldg.merge(bldg_class, how="left", on="full_plus_code")
bldg = bldg.loc[bldg["bldg_class"]==1].reset_index(drop=True)

In [4]:
city = gpd.read_file("..//data//ncr//adm_bounds.gpkg", layer="city").to_crs(crs="EPSG:32651")
brgy = gpd.read_file("..//data//ncr//adm_bounds.gpkg", layer="barangay").to_crs(crs="EPSG:32651")
brgy = brgy.loc[~brgy["ADM4_EN"].isin(["Manila North Cemetery","Tutuban Mall (Claimed by Five Barangays of Tondo, Manila)"])].reset_index(drop=True)

In [5]:
bldg["area"] = bldg.area
bldg['centroid'] = bldg.geometry.centroid
bldg_cent = bldg.set_geometry("centroid")

bldg_det = bldg_cent.sjoin(city[["ADM3_EN","ADM3_PCODE","geometry","pop"]],how="left",predicate="within")\
        .drop(columns=["index_right"])\
        .sjoin(brgy[["ADM4_EN","ADM4_PCODE","ADM3_EN","geometry"]],how="left",predicate="within")\
        .drop(columns=["index_right"])

def get_pop_per_area(df):
    return df["area"]*(df["pop"].values[0]/df["area"].sum())

bldg_det["pop_est"] = bldg_det.groupby("ADM3_PCODE",as_index=False).apply(get_pop_per_area)

brgy_fin = brgy.merge(bldg_det[["ADM4_PCODE","pop_est"]].groupby("ADM4_PCODE",as_index=False).sum(),how="left",on="ADM4_PCODE")
brgy_fin.loc[pd.isna(brgy_fin.pop_est),"pop_est"] = 0

brgy_fin.to_file("..//data//ncr//brgy_pop_estimates.gpkg",driver="GPKG",layer="building_based_filtered")
brgy_fin[["ADM4_EN","ADM3_EN","ADM4_PCODE","pop","pop_est"]].to_csv("..//pop_estimates//ncr_building_based_filtered.csv",index=False)